# Làm sạch dữ liệu và xây kho dữ liệu (mô hình sao)

**Đầu vào:** bảng `staging_retail` trong PostgreSQL (dữ liệu thô).
**Đầu ra:** 4 bảng chiều (`dim_date`, `dim_country`, `dim_product`, `dim_customer`) và 2 bảng sự kiện (`fact_sales`, `fact_returns`).

**Ba nguyên tắc làm sạch (rút ra từ bước EDA):**
1. Không xóa mù quáng: tách giao dịch thành *đơn bán* và *đơn hoàn/hủy*, ghi lại số dòng bị loại ở mỗi bước.
2. Dòng **thiếu Customer ID không bị xóa**: gắn vào khách `Unknown` (`customer_key = 0`) để tổng doanh thu vẫn khớp dữ liệu gốc. Khi phân khúc và làm mô hình thì mới loại khách này.
3. Đơn sỉ lớn là hợp lệ nên **giữ nguyên**; chỉ loại những dòng sai/không phải bán hàng.

In [1]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

pd.options.display.float_format = "{:,.2f}".format

# Kết nối PostgreSQL (mật khẩu nhập khi chạy, không ghi trong file)
engine = create_engine(URL.create(
    "postgresql+psycopg2", username="postgres", password=getpass("Mật khẩu PostgreSQL: "),
    host="localhost", port=5432, database="Churn_Data"))

Mật khẩu PostgreSQL:  ········


## 1. Đọc dữ liệu

In [2]:
df = pd.read_sql("SELECT * FROM staging_retail", engine)
print("Số dòng trong staging_retail:", len(df))
df.head()

Số dòng trong staging_retail: 1067371


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Chuẩn hóa tên cột và kiểu dữ liệu (chạy được dù staging nạp theo cách cũ hay mới)
df = df.rename(columns={"Invoice": "invoice", "StockCode": "stock_code", "Description": "description",
                        "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "price",
                        "Customer ID": "customer_id", "Country": "country"})

df["invoice"] = df["invoice"].astype(str)
df["stock_code"] = df["stock_code"].astype(str).str.strip().str.upper()
df["description"] = df["description"].str.strip()
df["country"] = df["country"].str.strip()
df["invoice_ts"] = pd.to_datetime(df["invoice_date"])                                   # ngày giờ giao dịch
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")  # số nguyên, cho phép trống
df = df.drop(columns="invoice_date")
df.dtypes

invoice                   str
stock_code                str
description               str
quantity                int64
price                 float64
customer_id             Int64
country                   str
invoice_ts     datetime64[us]
dtype: object

## 2. Làm sạch từng bước

In [4]:
# Bước 1: xóa dòng trùng lặp hoàn toàn
truoc = len(df)
df = df.drop_duplicates()
print("Bước 1 – đã xóa", truoc - len(df), "dòng trùng")

Bước 1 – đã xóa 34335 dòng trùng


In [5]:
# Bước 2: bỏ hóa đơn bắt đầu bằng chữ A (điều chỉnh nợ xấu, không phải giao dịch bán)
truoc = len(df)
df = df[~df["invoice"].str.startswith("A")]
print("Bước 2 – đã bỏ", truoc - len(df), "dòng")

Bước 2 – đã bỏ 6 dòng


In [6]:
# Bước 3: chỉ giữ mã sản phẩm thật (bắt đầu bằng 5 chữ số). Loại POST, DOT, M, BANK CHARGES, AMAZONFEE...
truoc = len(df)
df = df[df["stock_code"].str.match(r"^\d{5}")]
print("Bước 3 – đã bỏ", truoc - len(df), "dòng")

Bước 3 – đã bỏ 5974 dòng


In [7]:
# Bước 4: tách đơn hoàn/hủy (hóa đơn bắt đầu bằng C) khỏi đơn bán
la_huy = df["invoice"].str.startswith("C")
sales = df[~la_huy].copy()
returns = df[la_huy].copy()
print("Đơn bán:", len(sales), "dòng | Đơn hoàn/hủy:", len(returns), "dòng")

Đơn bán: 1009142 dòng | Đơn hoàn/hủy: 17914 dòng


In [8]:
# Bước 5: đơn bán – bỏ dòng số lượng <= 0 hoặc giá <= 0 (điều chỉnh kho, hàng hỏng/mất có giá 0)
truoc = len(sales)
sales = sales[(sales["quantity"] > 0) & (sales["price"] > 0)]
print("Bước 5 – đã bỏ", truoc - len(sales), "dòng đơn bán")

Bước 5 – đã bỏ 5928 dòng đơn bán


In [9]:
# Bước 6: đơn hoàn/hủy – số lượng đang âm nên đổi sang số dương, bỏ dòng giá <= 0
returns["quantity"] = returns["quantity"].abs()
truoc = len(returns)
returns = returns[(returns["quantity"] > 0) & (returns["price"] > 0)]
print("Bước 6 – đã bỏ", truoc - len(returns), "dòng đơn hoàn/hủy")

Bước 6 – đã bỏ 0 dòng đơn hoàn/hủy


**Bước 7 – Ghép cặp bán–hủy.** Có những đơn nhập nhầm rồi hủy ngay (ví dụ 80.995 và 74.215 sản phẩm). Nếu giữ lại, các đơn này làm lệch doanh thu và top sản phẩm.
Quy tắc: *cùng khách, cùng mã sản phẩm, cùng số lượng, đơn hủy xảy ra trong vòng 1 ngày sau đơn bán* thì coi là nhập nhầm và loại cả hai. Chỉ ghép được với giao dịch có Customer ID.

In [10]:
# Bảng đơn bán và đơn hủy (chỉ lấy dòng có Customer ID), giữ lại số thứ tự dòng gốc để xóa sau
ban = sales.dropna(subset=["customer_id"])[["customer_id", "stock_code", "quantity", "invoice_ts"]].reset_index(names="dong_ban")
huy = returns.dropna(subset=["customer_id"])[["customer_id", "stock_code", "quantity", "invoice_ts"]].reset_index(names="dong_huy")

# Ghép theo khách + mã + số lượng, rồi chỉ giữ các cặp hủy xảy ra trong 1 ngày sau khi bán
cap = ban.merge(huy, on=["customer_id", "stock_code", "quantity"], suffixes=("_ban", "_huy"))
cap = cap[(cap["invoice_ts_huy"] >= cap["invoice_ts_ban"]) &
          (cap["invoice_ts_huy"] <= cap["invoice_ts_ban"] + pd.Timedelta(days=1))]

# Mỗi đơn hủy chỉ ghép với 1 đơn bán gần nhất, mỗi đơn bán chỉ ghép 1 lần
cap = cap.sort_values("invoice_ts_ban", ascending=False).drop_duplicates("dong_huy").drop_duplicates("dong_ban")

sales = sales.drop(index=cap["dong_ban"]).reset_index(drop=True)
returns = returns.drop(index=cap["dong_huy"]).reset_index(drop=True)
print("Bước 7 – đã loại", len(cap), "dòng bán và", len(cap), "dòng hủy tương ứng")

Bước 7 – đã loại 1331 dòng bán và 1331 dòng hủy tương ứng


### Dòng thiếu Customer ID
Không xóa. Xem tỷ trọng doanh thu của nhóm này để thấy vì sao phải giữ:

In [11]:
thieu_id = sales["customer_id"].isna()
doanh_thu = sales["quantity"] * sales["price"]
print(f"Dòng đơn bán thiếu Customer ID: {thieu_id.sum():,} ({thieu_id.mean() * 100:.1f}% số dòng)")
print(f"Tỷ trọng doanh thu của nhóm này: {doanh_thu[thieu_id].sum() / doanh_thu.sum() * 100:.1f}%")

Dòng đơn bán thiếu Customer ID: 226,637 (22.6% số dòng)
Tỷ trọng doanh thu của nhóm này: 13.4%


## 3. Tạo các bảng chiều

In [12]:
tat_ca = pd.concat([sales, returns])      # gộp để mọi mã/khách đều có trong bảng chiều

# --- dim_date: mỗi dòng một ngày ---
ngay = pd.date_range(tat_ca["invoice_ts"].min().normalize(), tat_ca["invoice_ts"].max().normalize(), freq="D")
dim_date = pd.DataFrame({"full_date": ngay})
dim_date["year_no"] = dim_date["full_date"].dt.year
dim_date["quarter_no"] = dim_date["full_date"].dt.quarter                                                
dim_date["month_no"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["year_quarter"] = dim_date["year_no"].astype(str) + "-Q" + dim_date["quarter_no"].astype(str)   # ví dụ 2010-Q4
dim_date["year_month"] = dim_date["full_date"].dt.strftime("%Y-%m")                                      # ví dụ 2010-12
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["full_date"] = dim_date["full_date"].dt.date
dim_date.head(3)

,full_date,year_no,quarter_no,month_no,month_name,year_quarter,year_month,day_name
0,2009-12-01,2009,4,12,December,2009-Q4,2009-12,Tuesday
1,2009-12-02,2009,4,12,December,2009-Q4,2009-12,Wednesday
2,2009-12-03,2009,4,12,December,2009-Q4,2009-12,Thursday


In [13]:
# --- dim_product: mỗi mã sản phẩm một dòng; mô tả = mô tả xuất hiện nhiều nhất của mã đó ---
mo_ta = tat_ca.dropna(subset=["description"]).groupby("stock_code")["description"].agg(lambda s: s.mode().iloc[0])
dim_product = pd.DataFrame({"stock_code": sorted(tat_ca["stock_code"].unique())})
dim_product["description"] = dim_product["stock_code"].map(mo_ta)
dim_product.head(3)

,stock_code,description
0,10002,INFLATABLE POLITICAL GLOBE
1,10002R,ROBOT PENCIL SHARPNER
2,10080,GROOVY CACTUS INFLATABLE


In [14]:
# --- dim_customer: mỗi khách một dòng. customer_key chính là mã khách hàng gốc (Customer ID);
#     customer_key = 0 là khách "Unknown" dành cho các giao dịch thiếu Customer ID ---
nuoc_chinh = tat_ca.dropna(subset=["customer_id"]).groupby("customer_id")["country"].agg(lambda s: s.mode().iloc[0])
mua_dau = sales.dropna(subset=["customer_id"]).groupby("customer_id")["invoice_ts"].min().dt.date

dim_customer = pd.DataFrame({"customer_key": nuoc_chinh.index.astype("int64"), "country": nuoc_chinh.values})
dim_customer["country_group"] = "Export"                                                   # UK / Export / Unknown
dim_customer.loc[dim_customer["country"] == "United Kingdom", "country_group"] = "UK"
dim_customer.loc[dim_customer["country"] == "Unspecified", "country_group"] = "Unknown"
dim_customer["first_purchase_date"] = dim_customer["customer_key"].map(mua_dau)           # ngày mua đầu tiên

unknown = pd.DataFrame({"customer_key": [0], "country": [None], "country_group": ["Unknown"],
                        "first_purchase_date": [None]})
dim_customer = pd.concat([unknown, dim_customer], ignore_index=True)
dim_customer.head(3)

,customer_key,country,country_group,first_purchase_date
0,0,None,Unknown,None
1,12346,United Kingdom,UK,2010-03-02
2,12347,Iceland,Export,2010-10-31


## 4. Tạo các bảng sự kiện

In [15]:
cot = ["invoice_no", "order_date", "customer_key", "stock_code", "country", "quantity", "unit_price"]

# Đơn bán: thêm ngày, mã khách (thiếu ID -> 0) và doanh thu
fact_sales = sales.rename(columns={"invoice": "invoice_no", "price": "unit_price"})
fact_sales["order_date"] = fact_sales["invoice_ts"].dt.date
fact_sales["customer_key"] = fact_sales["customer_id"].fillna(0).astype(int)
fact_sales["revenue"] = (fact_sales["quantity"] * fact_sales["unit_price"]).round(4)
fact_sales = fact_sales[cot + ["revenue"]]

# Đơn hoàn/hủy: làm tương tự
fact_returns = returns.rename(columns={"invoice": "invoice_no", "price": "unit_price"})
fact_returns["order_date"] = fact_returns["invoice_ts"].dt.date
fact_returns["customer_key"] = fact_returns["customer_id"].fillna(0).astype(int)
fact_returns["return_value"] = (fact_returns["quantity"] * fact_returns["unit_price"]).round(4)
fact_returns = fact_returns[cot + ["return_value"]]

print("fact_sales:", fact_sales.shape, "| fact_returns:", fact_returns.shape)
fact_sales.head(3)

fact_sales: (1001883, 8) | fact_returns: (16583, 8)


,invoice_no,order_date,customer_key,stock_code,country,quantity,unit_price,revenue
0,489434,2009-12-01,13085,85048,United Kingdom,12,6.95,83.40
1,489434,2009-12-01,13085,79323P,United Kingdom,12,6.75,81.00
2,489434,2009-12-01,13085,79323W,United Kingdom,12,6.75,81.00


## 5. Tạo bảng trong PostgreSQL và nạp dữ liệu
Chạy lại notebook sẽ xóa và tạo lại 5 bảng. Bảng `staging_retail` (dữ liệu thô) không dùng đến.
Khóa chính ở bảng chiều và khóa ngoại ở bảng sự kiện giúp PostgreSQL tự kiểm tra dữ liệu và cho Power BI thấy quan hệ giữa các bảng.

In [16]:
tao_bang = """
DROP TABLE IF EXISTS fact_returns CASCADE;
DROP TABLE IF EXISTS fact_sales CASCADE;
DROP TABLE IF EXISTS dim_customer CASCADE;
DROP TABLE IF EXISTS dim_product CASCADE;
DROP TABLE IF EXISTS dim_date CASCADE;

CREATE TABLE dim_date (
    full_date     DATE PRIMARY KEY,
    year_no       SMALLINT,
    quarter_no    SMALLINT,
    month_no      SMALLINT,
    month_name    TEXT,
    year_quarter  TEXT,
    year_month    TEXT,
    day_name      TEXT
);

CREATE TABLE dim_product (
    stock_code    TEXT PRIMARY KEY,
    description   TEXT
);

CREATE TABLE dim_customer (
    customer_key         INTEGER PRIMARY KEY,   -- mã khách hàng gốc; 0 = Unknown (thiếu Customer ID)
    country              TEXT,
    country_group        TEXT,                  -- UK / Export / Unknown
    first_purchase_date  DATE
);

CREATE TABLE fact_sales (
    invoice_no    TEXT,
    order_date    DATE    REFERENCES dim_date (full_date),
    customer_key  INTEGER REFERENCES dim_customer (customer_key),
    stock_code    TEXT    REFERENCES dim_product (stock_code),
    country       TEXT,
    quantity      INTEGER,
    unit_price    NUMERIC(12,4),
    revenue       NUMERIC(14,4)
);

CREATE TABLE fact_returns (
    invoice_no    TEXT,
    order_date    DATE    REFERENCES dim_date (full_date),
    customer_key  INTEGER REFERENCES dim_customer (customer_key),
    stock_code    TEXT    REFERENCES dim_product (stock_code),
    country       TEXT,
    quantity      INTEGER,
    unit_price    NUMERIC(12,4),
    return_value  NUMERIC(14,4)
);
"""

with engine.begin() as conn:
    conn.exec_driver_sql(tao_bang)
print("Đã tạo 5 bảng của mô hình sao")

Đã tạo 5 bảng của mô hình sao


In [17]:
# Nạp bảng chiều trước, bảng sự kiện sau (vì bảng sự kiện tham chiếu bảng chiều).
cac_bang = {"dim_date": dim_date, "dim_product": dim_product, "dim_customer": dim_customer,
            "fact_sales": fact_sales, "fact_returns": fact_returns}

for ten, bang in cac_bang.items():
    bang.to_sql(ten, engine, if_exists="append", index=False, chunksize=10000, method="multi")
    print(f"Đã nạp {ten}: {len(bang):,} dòng")

Đã nạp dim_date: 739 dòng
Đã nạp dim_product: 4,717 dòng
Đã nạp dim_customer: 5,875 dòng
Đã nạp fact_sales: 1,001,883 dòng
Đã nạp fact_returns: 16,583 dòng


## 6. Đối soát: số liệu

In [19]:
kq = []
for ten, bang in cac_bang.items():
    n_db = int(pd.read_sql(f"SELECT COUNT(*) AS n FROM {ten}", engine)["n"][0])
    kq.append({"bang": ten, "dong_trong_notebook": len(bang), "dong_trong_postgres": n_db, "khop": len(bang) == n_db})

kq = pd.DataFrame(kq)
assert kq["khop"].all(), "Có bảng không khớp, kiểm tra lại bước nạp"
kq

,bang,dong_trong_notebook,dong_trong_postgres,khop
0,dim_date,739,739,True
1,dim_product,4717,4717,True
2,dim_customer,5875,5875,True
3,fact_sales,1001883,1001883,True
4,fact_returns,16583,16583,True


## 7. Thử truy vấn trên mô hình sao
Nối bảng sự kiện với hai bảng chiều: doanh thu theo quý và nhóm khách (UK / Export / Unknown). Nhóm `Unknown` chủ yếu là các giao dịch thiếu Customer ID.

In [20]:
truy_van = """
SELECT d.year_quarter, c.country_group,
       COUNT(DISTINCT f.invoice_no) AS so_hoa_don,
       ROUND(SUM(f.revenue))        AS doanh_thu
FROM fact_sales f
JOIN dim_date     d ON f.order_date   = d.full_date
JOIN dim_customer c ON f.customer_key = c.customer_key
GROUP BY d.year_quarter, c.country_group
ORDER BY d.year_quarter, c.country_group
"""
pd.read_sql(truy_van, engine).astype({"doanh_thu": float})

,year_quarter,country_group,so_hoa_don,doanh_thu
0,2009-Q4,Export,79,"69,168.00"
1,2009-Q4,UK,1413,"605,892.00"
2,2009-Q4,Unknown,169,"119,822.00"
3,2010-Q1,Export,279,"308,314.00"
4,2010-Q1,UK,3268,"1,385,943.00"
5,2010-Q1,Unknown,326,"215,241.00"
6,2010-Q2,Export,313,"269,964.00"
7,2010-Q2,UK,3829,"1,510,503.00"
8,2010-Q2,Unknown,373,"179,728.00"
9,2010-Q3,Export,343,"303,503.00"


## 8. Tổng kết 

Khách Unknown (customer_key = 0) vẫn có trong fact_sales để tổng doanh thu khớp, nhưng sẽ bị loại khỏi phân khúc và mô hình.

| Bước (`buoc`) | Số dòng (`so_dong`) |
| :--- | :--- |
| 0. Dữ liệu staging ban đầu | 1067371 |
| 1. Dòng trùng lặp hoàn toàn | 34335 |
| 2. Hóa đơn điều chỉnh nợ xấu (tiền tố A) | 6 |
| 3. Mã không phải sản phẩm | 5974 |
| 4. Đơn bán có số lượng <= 0 hoặc giá <= 0 | 5928 |
| 5. Đơn hoàn/hủy có giá <= 0 | 0 |
| 6. Cặp bán–hủy trong 1 ngày (loại cả hai phía) | 2662 |
| Kết quả: số dòng đơn bán hợp lệ | 1001883 |
| Kết quả: số dòng đơn hoàn/hủy hợp lệ | 16583 | 16583 |it